# Clipora AI Presenter — Free Nepali Generator

This notebook is the heavy-AI companion for Clipora. It uses the free Colab GPU when available. No paid API key is required.

**Pipeline:** Clipora package → Nepali Chatterbox TTS → SadTalker → FFmpeg → MP4.

Only process images and voices you have permission to use. Colab free GPU availability is not guaranteed.

## 1. Upload the Clipora presenter package

In Clipora, use **AI Presenter → Create package**, then upload the downloaded `.zip` here. If you only have a script/image/voice, you can upload those manually instead.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded))

In [ ]:
import os, zipfile, json, shutil, pathlib
WORK='/content/clipora_presenter'
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)
name=next(iter(uploaded))
if name.lower().endswith('.zip'):
    with zipfile.ZipFile(name) as z: z.extractall(WORK)
else:
    shutil.copy(name, os.path.join(WORK,name))

job_path=None
for root,dirs,fs in os.walk(WORK):
    for f in fs:
        if f.endswith('.json') and 'job' in f.lower(): job_path=os.path.join(root,f)
if job_path:
    job=json.load(open(job_path,encoding='utf-8'))
    print(json.dumps(job,ensure_ascii=False,indent=2))
else:
    job={'input':{'script':'नमस्ते! यो Clipora AI Presenter हो।','aspect':'9:16'}}
    print('No job JSON found; using demo defaults.')

## 2. Install Nepali TTS

The current Nepali Chatterbox model provides a T4/free-tier Colab workflow and supports optional short reference-voice cloning.

In [ ]:
!pip -q install git+https://github.com/Imbatmann/chatterbox-nepali.git safetensors librosa

In [ ]:
import os, glob, torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

device='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',device)
model=ChatterboxMultilingualTTS.from_pretrained(device)
ckpt=hf_hub_download('Imbatmann/chatterbox-nepali-tts','t3_mtl_nepali_final.safetensors')
sd=load_file(ckpt)
cleaned={k.replace('patched_model.','').replace('model.',''):v for k,v in sd.items()}
model.t3.load_state_dict(cleaned,strict=False)
model.t3.to(device).eval()
script=job.get('input',{}).get('script','नमस्ते, यो Clipora AI Presenter हो।')
refs=[]
for root,dirs,fs in os.walk(WORK):
    refs += [os.path.join(root,f) for f in fs if f.lower().endswith(('.wav','.mp3','.m4a','.flac'))]
ref=refs[0] if refs else None
kwargs={'exaggeration':0.5,'temperature':0.8}
if ref: kwargs['audio_prompt_path']=ref
wav=model.generate(script,'ne',**kwargs)
TTS_WAV='/content/output.wav'
torchaudio.save(TTS_WAV,wav,model.sr)
print('Created:',TTS_WAV)

## 3. Install SadTalker and models

SadTalker turns one portrait image plus the generated audio into a talking-head video. The project is Apache 2.0 licensed and provides model-download scripts and CLI/WebUI workflows.

In [ ]:
%cd /content
!git clone -q https://github.com/OpenTalker/SadTalker.git
%cd /content/SadTalker
!pip -q install -r requirements.txt
!bash scripts/download_models.sh

In [ ]:
import os
images=[]
for root,dirs,fs in os.walk(WORK):
    images += [os.path.join(root,f) for f in fs if f.lower().endswith(('.png','.jpg','.jpeg','.webp'))]
if not images: raise RuntimeError('No presenter image found in the uploaded package.')
IMAGE=images[0]
RESULT='/content/clipora_result'
os.makedirs(RESULT,exist_ok=True)
print('Presenter:',IMAGE)
!python inference.py --driven_audio /content/output.wav --source_image "{IMAGE}" --result_dir /content/clipora_result --still --preprocess full

## 4. Make the final Clipora MP4

In [ ]:
import glob, subprocess, os
videos=glob.glob('/content/clipora_result/*.mp4')
if not videos: raise RuntimeError('SadTalker did not create an MP4. Check the output above.')
src=videos[-1]
aspect=job.get('output',{}).get('aspect',job.get('input',{}).get('aspect','9:16'))
if aspect=='16:9': vf='scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2'
elif aspect=='1:1': vf='scale=1080:1080:force_original_aspect_ratio=decrease,pad=1080:1080:(ow-iw)/2:(oh-ih)/2'
else: vf='scale=1080:1920:force_original_aspect_ratio=decrease,pad=1080:1920:(ow-iw)/2:(oh-ih)/2'
OUT='/content/clipora-presenter.mp4'
subprocess.run(['ffmpeg','-y','-i',src,'-vf',vf,'-r','30','-c:v','libx264','-c:a','aac','-movflags','+faststart',OUT],check=True)
print('FINAL:',OUT)

In [ ]:
from google.colab import files
files.download('/content/clipora-presenter.mp4')